# Baseline: gated ABMIL on the packed tissue set

**The question:** what does our model score, trained from the packed feature
set, is it the same model as Stage 9's, and how much of the cross-site score is
noise?

The model is notebook 9's gated ABMIL with its fixed settings, never tuned
(hidden 512, attention 256, dropout 0.25, AdamW 1e-4, weight decay 1e-5,
cosine, 20 epochs, cross-entropy, no early stopping), now in `src/mil.py`. It
is the baseline every later change is measured against, starting with stain
normalisation (notebooks 13 and 14).

Protocols:

| Protocol | Train | Test | Seeds |
|---|---|---|---|
| mixed | four Stage 4 folds | the fifth, pooled over the five | 0 |
| Radboud to Karolinska | every Radboud slide | every Karolinska slide | 0 to 4 |
| Karolinska to Radboud | every Karolinska slide | every Radboud slide | 0 to 4 |
| Radboud to Karolinska, fold-based | Radboud slides of four folds | Karolinska slides of the fifth | 0 |

The last is Stage 9's own cross-site protocol, kept only to prove the module
reproduces Stage 9. Cross-site runs no longer use folds (no duplicate group
spans both hospitals, so nothing can leak), and run in both directions with five
seeds: a single direction and seed cannot tell a real change from noise.
Every score comes with the predicted-grade distribution, because the two
hospitals' grades differ and a cross-site score can move just by shifting
predictions up or down.

**It must reproduce Stage 9:** 0.864 mixed (0.881 Karolinska, 0.824 Radboud)
and 0.596 fold-based Radboud to Karolinska. Training repeats exactly on one GPU
(FINDINGS, Stage 9) and the packed rows equal the per-slide files exactly
(Stage 11), so any difference is a bug.

## Preamble

In [1]:
"""Make the project's src/ folder importable.

Locally it sits one folder up from notebooks/. On Kaggle it arrives as an
attached dataset, so every attached dataset is checked for a src/ folder, and
the notebook stops with a clear message if there is none.
"""

import glob
import os
import sys

SRC_ROOT = next(
    (
        root
        for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
        if os.path.isfile(os.path.join(root, "src", "tiling.py"))
    ),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

In [2]:
"""Imports.

time                  timing each protocol.
pandas                labels, predictions and score tables.
tqdm                  progress over folds and seeds.
src.features          the packed feature set from Stage 11.
src.mil               the model, the protocols, scores and the saved predictions.
src.paths             where folds.csv is, locally and on Kaggle.
"""

import time

import pandas as pd
from tqdm.auto import tqdm

from src.features import (
    open_packed,  # opens a packed set: features memory-mapped read-only [tiles, 1536], index (slide_id, start, end, n_tiles), starts, ends, tiles table, manifest; slide i is features[starts[i]:ends[i]]
)
from src.mil import (
    CROSS_SITE,  # the two cross-site directions: name -> (training hospital, test hospital)
    EPOCHS,  # 20 training epochs, no early stopping
    FOLD_PROTOCOLS,  # "mixed" and Stage 9's "radboud_to_karolinska_folds", both on the Stage 4 folds
    SEEDS,  # 0 to 4: one model per seed in each cross-site direction
    cross_site,  # train on every slide of one hospital, test on every slide of the other, once per seed; one prediction column per seed
    cross_validate,  # the five folds under a fold protocol; pooled test predictions
    grade_distribution,  # share of slides per grade, true against predicted: shows over- or under-grading
    load_bags,  # every slide's features from the packed set, copied into RAM, by slide_id
    save_predictions,  # writes predictions with grade, hospital and fold to <version>/predictions/<name>.parquet
    score_table,  # one row per protocol and slide group: n and QWK (seed mean and spread for cross-site)
)
from src.paths import (
    DATA_DIR,  # the competition data folder (data/ locally, /kaggle/input/... on Kaggle); folds.csv lives here locally
    find_input,  # the path of a file made earlier; prints where it found it; stops with a clear error if missing
)

In [3]:
"""Settings, labels and the packed set.

1. VERSION is the feature set the baseline trains on: the packed tissue set.
2. STAGE9 holds the scores FINDINGS records for the same model trained from the
   per-slide files (Stage 9, all slides), by protocol and slide group. The
   Stage 9 protocols here must reproduce them to three decimals.
3. The model's settings and seeds live in src/mil.py, one copy only.
4. labels is the Stage 4 fold file limited to the packed slides, which leaves
   out the excluded empty slide exactly as Stage 9 did. It is printed per fold
   and hospital, and as each hospital's grade distribution, because that
   difference is what the cross-site directions have to bridge.
"""

VERSION = "uni2h-L1-224-t144-hsv-v1"
STAGE9 = {
    ("mixed", "all"): 0.864,
    ("mixed", "karolinska"): 0.881,
    ("mixed", "radboud"): 0.824,
    ("radboud_to_karolinska_folds", "all"): 0.596,
}

packed = open_packed(VERSION)
folds = pd.read_csv(find_input("folds.csv", local_dir=DATA_DIR)).set_index("slide_id")
labels = folds.loc[packed.index["slide_id"]]
print(f"{VERSION}: {len(labels):,} slides, excluded {list(packed.manifest['packed']['excluded_slides'])}; "
      f"{EPOCHS} epochs, cross-site seeds {SEEDS}")
print(pd.crosstab(labels["fold"], labels["provider"]).to_string())
print("\ngrade share per hospital:")
print(pd.crosstab(labels["provider"], labels["isup_grade"], normalize="index").round(3).to_string())

folds.csv: /home/aztrek/projects/Prostate-Cancer-Grading/data/folds.csv
uni2h-L1-224-t144-hsv-v1: 10,615 slides, excluded ['3790f55cad63053e956fb73027179707']; 20 epochs, cross-site seeds [0, 1, 2, 3, 4]
provider  karolinska  radboud
fold                         
0               1092     1031
1               1090     1033
2               1091     1031
3               1091     1033
4               1091     1032

grade share per hospital:
isup_grade      0      1      2      3      4      5
provider                                            
karolinska  0.353  0.333  0.122  0.058  0.088  0.046
radboud     0.187  0.165  0.131  0.179  0.149  0.189


## Load the features

Every slide's rows are copied out of the memory-mapped array once (2.1 GB), so
training reads RAM, not disk.

In [4]:
"""Load every packed slide's features into memory, timed."""

started = time.perf_counter()
bags = load_bags(packed)
print(f"loaded {len(bags):,} bags in {time.perf_counter() - started:.0f} s")

loaded 10,615 bags in 1 s


## Does it reproduce Stage 9?

Both of Stage 9's protocols on the Stage 4 folds, seed 0. The notebook stops if
any score differs from Stage 9's at three decimals.

In [5]:
"""Run Stage 9's two fold protocols and compare every score with STAGE9.

1. cross_validate trains one model per fold and pools the test predictions.
2. score_table scores them, mixed also per hospital.
3. Each (protocol, slides) score is rounded to three decimals and compared with
   STAGE9; any difference stops the notebook.
"""

predictions = {}
for protocol in FOLD_PROTOCOLS:
    started = time.perf_counter()
    predictions[protocol] = cross_validate(bags, labels, protocol, progress=tqdm)
    print(f"{protocol}: {(time.perf_counter() - started) / 60:.1f} min")

reproduction = score_table(labels, {p: predictions[p] for p in FOLD_PROTOCOLS})
reproduction["stage 9"] = [STAGE9.get((p, s)) for p, s in zip(reproduction["protocol"], reproduction["slides"])]
print(reproduction[["protocol", "slides", "n", "qwk", "stage 9"]].round(4).to_string(index=False))
different = reproduction[reproduction["qwk"].round(3) != reproduction["stage 9"]]
if len(different):
    raise RuntimeError(f"does not reproduce Stage 9:\n{different.to_string(index=False)}")
print("Stage 9 reproduced exactly")

mixed: 11.4 min
radboud_to_karolinska_folds: 5.2 min
                   protocol     slides     n    qwk  stage 9
                      mixed        all 10615 0.8643    0.864
                      mixed    radboud  5160 0.8240    0.824
                      mixed karolinska  5455 0.8807    0.881
radboud_to_karolinska_folds        all  5455 0.5959    0.596
Stage 9 reproduced exactly


## Cross-site, both directions, five seeds

Train on every slide of one hospital, test on every slide of the other. Five
models per direction (seeds 0 to 4) show how much of the score is seed noise.

In [6]:
"""Train and predict each cross-site direction, once per seed."""

for name, (train_hospital, test_hospital) in CROSS_SITE.items():
    started = time.perf_counter()
    predictions[name] = cross_site(bags, labels, train_hospital, test_hospital, progress=tqdm)
    print(f"{name}: {(time.perf_counter() - started) / 60:.1f} min")

radboud_to_karolinska: 6.5 min
karolinska_to_radboud: 7.1 min


## Scores and predicted grades

QWK for every protocol: mixed overall and per hospital; each cross-site
direction as the mean over five seeds with the spread between them. Then, per
protocol, the share of slides predicted at each grade against the true share:
a cross-site score that collapses because the model over-grades the other
hospital shows up here.

In [7]:
"""Score every protocol, then show the predicted-grade distribution of each."""

scores = score_table(labels, predictions)
print(scores.round(4).to_string(index=False))

for protocol, predicted in predictions.items():
    truth = labels.loc[predicted.index, "isup_grade"]
    print(f"\n{protocol} ({len(predicted):,} slides): share per grade")
    print(grade_distribution(truth, predicted).T.round(3).to_string())

                   protocol     slides     n  seeds    qwk    std    min    max
                      mixed        all 10615      1 0.8643 0.0000 0.8643 0.8643
                      mixed    radboud  5160      1 0.8240 0.0000 0.8240 0.8240
                      mixed karolinska  5455      1 0.8807 0.0000 0.8807 0.8807
radboud_to_karolinska_folds        all  5455      1 0.5959 0.0000 0.5959 0.5959
      radboud_to_karolinska        all  5455      5 0.6316 0.0201 0.6024 0.6563
      karolinska_to_radboud        all  5160      5 0.7800 0.0036 0.7754 0.7837

mixed (10,615 slides): share per grade
grade          0      1      2      3      4      5
true       0.272  0.251  0.127  0.117  0.118  0.115
predicted  0.290  0.255  0.122  0.112  0.108  0.113

radboud_to_karolinska_folds (5,455 slides): share per grade
grade          0      1      2      3      4      5
true       0.353  0.333  0.122  0.058  0.088  0.046
predicted  0.373  0.239  0.055  0.175  0.109  0.049

radboud_to_karolinska (5,4

## Save the predictions

Mixed and both cross-site directions, next to the feature set as
`predictions/abmil-<protocol>.parquet` (not tracked in git; rebuilt by running
this notebook). Notebook 14 compares every stain-normalised set with these,
slide for slide.

In [8]:
"""Save the predictions of mixed and of both cross-site directions."""

for protocol in ["mixed", *CROSS_SITE]:
    print(save_predictions(VERSION, f"abmil-{protocol}", labels, predictions[protocol]))

/home/aztrek/projects/Prostate-Cancer-Grading/data/features/uni2h-L1-224-t144-hsv-v1/predictions/abmil-mixed.parquet
/home/aztrek/projects/Prostate-Cancer-Grading/data/features/uni2h-L1-224-t144-hsv-v1/predictions/abmil-radboud_to_karolinska.parquet
/home/aztrek/projects/Prostate-Cancer-Grading/data/features/uni2h-L1-224-t144-hsv-v1/predictions/abmil-karolinska_to_radboud.parquet


## Result

Run 2026-11-17 on the RTX 4060 PC, tissue set `uni2h-L1-224-t144-hsv-v1`
(10,615 slides; the empty slide excluded).

**Stage 9 reproduces exactly.** The module, trained from the packed set, gives
Stage 9's scores to four decimals: mixed 0.8643 (Radboud 0.8240, Karolinska
0.8807) and fold-based Radboud to Karolinska 0.5959. Training took 11.4 and
5.2 minutes.

**Cross-site, every slide of one hospital to every slide of the other, five
seeds:**

| Direction | Test slides | QWK, mean of 5 seeds | Seed std | Min to max |
|---|---|---|---|---|
| Radboud to Karolinska | 5,455 | 0.632 | 0.020 | 0.602 to 0.656 |
| Karolinska to Radboud | 5,160 | 0.780 | 0.004 | 0.775 to 0.784 |

Training on every Radboud slide (rather than four folds' worth) raises Radboud
to Karolinska from 0.596 to 0.632 on average. Its seeds span 5 points, so a
single seed cannot judge a change of a few points; Karolinska to Radboud is
much steadier. 6.5 and 7.1 minutes for five models each.

**Predicted grades.** Mixed predictions follow the true shares closely. Across
hospitals the errors are not a general over-grading but a confusion of the
middle grades, different in each direction:
- Radboud to Karolinska: grade 2 predicted for 5.5% of slides (true 12.2%),
  grade 3 for 16.2% (true 5.8%).
- Karolinska to Radboud: grade 4 predicted for 29.6% (true 14.9%), grade 3
  for 9.6% (true 17.9%).

Predictions are saved as `predictions/abmil-mixed.parquet`,
`abmil-radboud_to_karolinska.parquet` and `abmil-karolinska_to_radboud.parquet`
in the feature set's folder: the baseline notebook 14 compares against.